# 2. Tiền xử lý dữ liệu

**Input:** `exps/data/data_EDA.csv`  
**Output:** `exps/feature0/` gồm `x_train.csv, y_train.csv, x_test.csv, test_id.csv, info.json`

BEGIN

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys; sys.path.insert(0, "../../extra")
from config import *
from common import *
from utils import *
display.clear_output()

In [3]:
MEMBER   = "baseline"      # tên/mã thành viên -> file log riêng
NOTEBOOK = "pre-processing"
FEATURE_SET = "feature0"   # thư mục output trong exps/
seed_everything()

In [4]:
df, numeric_columns, category_columns = load_eda()
df.shape

(2919, 82)

## 2.1. Làm sạch
Bỏ 4 cột thiếu nhiều, cột phân loại điền mode. Cột số để NaN — `make_pipeline()` ở stage 4 sẽ điền median **trong từng fold**.

In [5]:
DROP_COLS = ['Alley', 'PoolQC', 'Fence', 'MiscFeature']
df = df.drop(columns=DROP_COLS)
cat_cols = [c for c in category_columns if c not in DROP_COLS] + ['MSSubClass']
df['MSSubClass'] = df['MSSubClass'].astype(str)    # mã loại nhà, không phải số
for c in cat_cols:
    df[c] = df[c].fillna(df.loc[df.is_train == 1, c].mode()[0])   # mode tính trên train
df[cat_cols].isna().sum().sum()

np.int64(0)

## 2.2. Ngoại lai
Baseline: **chưa** loại.

In [6]:
outlier_mask = pd.Series(False, index=df.index)   # TODO: GrLivArea > 4000 (chỉ dòng train)
df = df[~outlier_mask].reset_index(drop=True)
print('Số dòng train:', df.is_train.sum())

Số dòng train: 1460


## 2.4. Mã hóa & target
One-hot trên train+test gộp (như code tham khảo) để hai tập có cùng cột.

In [7]:
y_train  = np.log1p(df.loc[df.is_train == 1, 'SalePrice'].values)
test_id  = df.loc[df.is_train == 0, 'Id'].values
X = pd.get_dummies(df.drop(columns=['Id', 'SalePrice']), columns=cat_cols, dtype=int)
x_train = X[X.is_train == 1].drop(columns='is_train').reset_index(drop=True)
x_test  = X[X.is_train == 0].drop(columns='is_train').reset_index(drop=True)
x_train.shape, x_test.shape

((1460, 289), (1459, 289))

## 2.5. Lưu

In [8]:
save_feature_set(FEATURE_SET, x_train, y_train, x_test, test_id,
                 info={'notebook': NOTEBOOK, 'member': MEMBER, 'note': 'mô tả ngắn thay đổi so với feature0'})

[save] d:/HocSau/House_Prices/house_prices_project/exps/feature0: x_train (1460, 289), x_test (1459, 289)


### Kết luận
- *(ghi nhận xét ở đây)*